# Course notebook — AI Software and Product Engineering

Use specifications, evaluations, secure development practices, and product discovery to deliver useful AI features.

Use Python 3.10 or later. All inputs are included and the workbook runs without API keys. Each lecture demonstration is a small model of the lecture mechanism. Run it, compare the result with your prediction, and use the section checkpoint to record what you learned.


## Practice case · The pull request nobody has time to review

Nadia’s scheduling team is asked to use a coding assistant to clear its backlog. The first demonstration creates a large change quickly. Reviewers now spend longer understanding the diffs and correcting edge cases. Nadia needs evidence that the entire workflow is getting better.

The workplace scenario is fictional.


## Discovery and specifications


### Lecture 01 · Discovering the Right Problem

Identify user jobs, workflow constraints, baseline performance, capability fit, failure cost, and measurable value before building.

#### Follow the wait through the whole workflow

A developer can draft a change in ten minutes, yet code sits three days awaiting a domain reviewer. More drafts make the queue worse.

Observe elapsed time from request to accepted change. If the constraint is review capacity, reduce diff size or automate evidence collection before paying to increase code output.

#### Read the mechanism

**user research**

User research observes real workflows, pain points, and workarounds before proposing AI features. It prevents building impressive demos nobody needs.

Watching support agents copy-paste from three systems reveals integration beats summarization.

**jobs-to-be-done**

Jobs-to-be-done frames what users hire a product to accomplish, not which technology it uses. AI fits when it improves the job outcome measurably.

Users hire expense tool to 'get reimbursed fast', not to 'chat with AI'.

**baseline workflow**

Baseline workflow documents how users solve the task today—time, errors, tools—before AI intervention. Improvement requires beating this baseline.

Manual ticket tagging takes 45s each; AI must beat accuracy and time with correction cost included.

**feasibility**

Feasibility assesses whether data, latency, risk, and model capability can meet requirements—not whether a demo works once.

If no labeled data exists and mistakes cost $10k, feasibility may be low despite flashy prototype.

**success metrics**

Success metrics tie releases to user-valued outcomes—task success, time saved, revenue—not model perplexity alone.

Deflect 20% of L1 tickets without increasing reopen rate defines success for support bot.

#### Worked example

**Situation:** A product team must convert a vague AI feature request into testable release evidence. Sales promised "AI onboarding assistant" without defining success.

**Baseline:** Build chat UI immediately—demo impresses but no measurable workflow improvement.

**Application:** Write problem brief: user job, current baseline workflow, failure costs, non-AI alternative, capability fit, success metrics (time-to-productive, error rate).

**Test cases:** (1) Normal: new hire with complete data. (2) Boundary: hire lacking manager assignment. (3) Adversarial: success metric gameable by skipping compliance steps.

**Measurement:** Baseline workflow timing study n≥20, projected ROI with confidence range, feasibility red flags.

**Design question:** What non-AI alternative would you ship if models were unavailable?

**Predict before running.** Optimize the human outcome, not the amount of AI in the product. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 9.1: Discovering the Right Problem"""

CHAPTER = "9.1"
print("chapter hook:", CHAPTER)
brief = {
    "job": "get employee productive day 1",
    "baseline_hours": 6.5,
    "failure_cost": "compliance breach",
    "non_ai": "checklist app with human verifier",
}
print(brief)
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Write a problem brief with a non-AI alternative.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why optimize human outcome rather than amount of AI?
2. How do gameable metrics create false success?
3. What discovery baseline skips user research?

<details><summary>Check your reasoning</summary>

Q1: AI is a means; value is workflow improvement. Q2: Metric improves while compliance worsens. Q3: Build because LLMs exist.

</details>


### Lecture 02 · Specification-Driven Development

Translate intent into functional, prompt, tool, agent, data, safety, and evaluation specifications with acceptance criteria.

#### Define what should happen before touching the code

A roster change should let users trade shifts, but managers ask who may confirm a swap. The example decision in the ticket leaves that open.

Write observable acceptance cases for both allowed and denied actions. Clarify which person owns the rule before implementation. Then use the tests to hold the assistant and human implementation to the same standard.

#### Read the mechanism

**functional specifications**

Functional specifications describe observable system behavior—inputs, outputs, errors—for builders and testers. They precede implementation and model choice.

Spec states: given valid invoice PDF, return JSON with vendor, total, date or structured error code.

**acceptance criteria**

Acceptance criteria are pass/fail conditions for feature completion—testable, unambiguous, tied to user value.

Given ambiguous date, system asks clarifying question rather than guessing—100% on test set.

**prompt specs**

Prompt specs version instructions, constraints, examples, and expected behaviors like API contracts. They enable review and regression unlike ad hoc prompts.

Prompt spec defines abstention when confidence low and JSON schema for outputs.

**tool contracts**

Tool contracts specify schemas, auth, idempotency, errors, and SLAs for each agent tool. They are integration boundaries models depend on.

search_docs contract promises p95 500ms, max 10 results, ReadScope auth.

**evaluation specs**

Evaluation specs define datasets, metrics, slices, and release thresholds before shipping. They turn 'good enough' into numbers.

Eval spec: 200 cases, faithfulness ≥ 0.9, P0 safety cases 100% pass.

#### Worked example

**Situation:** The onboarding assistant needs specs engineering and compliance can audit—prompts, tools, and evals must align.

**Baseline:** Slack thread of informal requirements.

**Application:** Write functional spec, prompt spec with acceptance examples, tool contracts, safety constraints, evaluation spec with executable pass/fail cases before coding.

**Test cases:** (1) Normal: hire with standard role. (2) Boundary: abstain when policy missing. (3) Adversarial: attempt privilege escalation via chat.

**Measurement:** Spec review sign-offs, % acceptance examples automated, defects found pre-impl vs post-impl.

**Design question:** Which acceptance example would fail if abstention behavior regresses?

**Predict before running.** Specifications align humans and agents around observable outcomes and constraints. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 9.2: Specification-Driven Development"""

acceptance = [
    {"input": "grant admin access", "expect": "require approval"},
    {"input": "unknown policy", "expect": "abstain"},
]
def check(outcome, expected):
    return expected in outcome
for case in acceptance:
    simulated = "abstain: no policy found" if "unknown" in case["input"] else "require approval"
    print(case["input"], check(simulated, case["expect"]))


#### Interpret and apply

Write executable examples before implementation.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why write executable examples before implementation?
2. What belongs in a tool contract versus a prompt?
3. What spec baseline is prose-only?

<details><summary>Check your reasoning</summary>

Q1: They define observable done and enable regression tests. Q2: Contracts: types, errors, auth; prompts: intent and format. Q3: Marketing one-pager without acceptance tests.

</details>


**Section checkpoint.** Write a baseline workflow and one measurable acceptance case before adding a model.


## Development and testing


### Lecture 03 · AI-Native Development Workflow

Organize repositories, instructions, skills, context files, branches, reviews, tests, and coding-agent collaboration.

#### Make the reviewer’s job concrete

The assistant proposes rewriting a scheduler and several unrelated files. The change is hard to validate, and nobody can tell which edit solved the original issue.

Use small proposals with a scoped diff, tests, and assumptions. Read the diff before expansion and ask for a separate change when behavior outside the agreed task would move.

#### Read the mechanism

**repo instructions**

Repo instructions—AGENTS.md, CONTRIBUTING—orient coding agents to build, test, and review conventions. They reduce wrong-file edits and skipped tests.

Instructions specify pytest command, lint rules, and forbidden directories.

**skills**

Skills package reusable agent capabilities—prompts, scripts, checklists—for specific tasks in Cursor and similar tools. They encode institutional workflow knowledge.

A 'create PR' skill runs tests, drafts description template, and calls gh CLI.

**context files**

Context files—.cursorrules, architecture docs—supply persistent project knowledge to coding agents. Stale context misleads worse than no context.

Architecture.md describes service boundaries so agent edits correct package.

**AI coding agents**

AI coding agents autonomously edit repositories given goals, tools, and constraints. They amplify throughput but require specs, tests, and human review.

Agent implements feature branch with tests; human reviews diff before merge.

**code review**

Code review evaluates correctness, security, and maintainability of changes—including agent-written code. It remains accountability gate before merge.

Reviewer checks agent did not skip auth on new endpoint despite passing happy-path tests.

#### Worked example

**Situation:** Team uses AI coding agents to implement onboarding; velocity rises but review burden spikes.

**Baseline:** Ad-hoc prompting in IDE with no repo instructions or test gates.

**Application:** Add AGENTS.md, skills for domain tasks, context files for architecture, require PR templates with eval evidence, compare review time across two assistant workflows.

**Test cases:** (1) Normal: bounded bugfix with tests. (2) Boundary: cross-module refactor. (3) Adversarial: agent adds silent dependency on deprecated API.

**Measurement:** PR review minutes, defect escape rate, test coverage delta.

**Design question:** Which repo instruction prevents agents from inventing nonexistent internal APIs?

**Predict before running.** AI accelerates change production, making specification and verification more important. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 9.3: AI-Native Development Workflow"""

CHAPTER = "9.3"
print("chapter hook:", CHAPTER)
REPO_RULES = ["run tests before commit", "use internal SDK docs", "no new deps without ADR"]
task = "add checkpoint resume"
checklist = [rule for rule in REPO_RULES]
print({"task": task, "agent_checklist": checklist})
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Run a bounded repository task with two assistants and compare review burden.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why does AI acceleration increase need for verification?
2. How do skills differ from generic repo rules?
3. What workflow baseline lacks repo instructions?

<details><summary>Check your reasoning</summary>

Q1: More code volume without spec/tests increases escape defects. Q2: Skills encode repeatable multi-step domain procedures. Q3: Blank repo with default Copilot only.

</details>


### Lecture 04 · Testing AI Systems

Combine unit, contract, integration, scenario, regression, adversarial, and human tests across deterministic and probabilistic components.

#### A passing example can leave a critical gap

The output shape is correct and happy-path tests pass. The assistant silently approves a duplicate shift swap on a retry.

Add a replay case and assert the business effect count. Keep fixture inputs and workflow versions fixed when comparing changes. Test that the race or retry does not simply disappear from logs.

#### Read the mechanism

**unit tests**

Unit tests verify deterministic functions and components in isolation with fast feedback. They anchor quality while model behavior stays statistical.

Parser unit tests cover edge cases agents might not consider when editing.

**contract tests**

Contract tests verify integrations between services—API schemas, tool responses—without full end-to-end runs. They catch breaking changes early.

Consumer test asserts search API returns fields reranker expects.

**eval datasets**

Eval datasets are labeled or rubric-scored cases representing production risks and happy paths. They must refresh as products and policies evolve.

200 support queries with gold answers updated quarterly after product launches.

**adversarial tests**

Adversarial tests probe injection, jailbreaks, edge inputs, and abuse scenarios. They belong in release gates for user-facing AI.

Prompt injection via ticket body attempting credential exfil must fail closed.

**release gates**

Release gates block deployment until eval, security, and performance criteria pass. They encode organizational risk tolerance numerically.

No deploy if faithfulness drops >2 points or p95 latency exceeds SLO.

#### Worked example

**Situation:** Onboarding assistant mixes deterministic validators and probabilistic generation; QA needs a coherent test strategy.

**Baseline:** Manual clicking in staging— misses regression on abstention behavior.

**Application:** Derive test pyramid: unit tests for validators, contract tests for tools, scenario tests for agent flows, eval dataset for language quality, adversarial injection cases.

**Test cases:** (1) Normal: schema validator unit test. (2) Boundary: flaky eval case near threshold. (3) Adversarial: prompt injection in integration scenario.

**Measurement:** Coverage by layer, release gate pass rate, escaped defects by test type.

**Design question:** Which layer catches abstention regression fastest and cheapest?

**Predict before running.** Test deterministic properties deterministically and probabilistic behavior statistically. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 9.4: Testing AI Systems"""

CHAPTER = "9.4"
print("chapter hook:", CHAPTER)
layers = ["unit", "contract", "scenario", "eval", "adversarial"]
catch_abstain = {"unit": False, "contract": False, "scenario": True, "eval": True, "adversarial": False}
for layer in layers:
    print(layer, "catches_abstain:", catch_abstain[layer])
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Derive a test pyramid from an AI system architecture.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. How do you test probabilistic vs deterministic components differently?
2. What belongs in release gates?
3. What testing baseline is manual only?

<details><summary>Check your reasoning</summary>

Q1: Deterministic: exact assertions; probabilistic: statistical eval over datasets. Q2: Automated scenarios plus eval thresholds block promote. Q3: QA clicks through UI before launch.

</details>


**Section checkpoint.** Turn a rejection behavior into a test that fails for a plausible but incorrect answer.


## Experience and value


### Lecture 05 · Human-Centered AI UX

Design uncertainty, citations, previews, correction, undo, approval, feedback, accessibility, and graceful failure.

#### A person needs an escape route

The new screen suggests a confident schedule but hides the reason a shift cannot be filled. A coordinator has to start over to make a manual correction.

Expose uncertain or rejected suggestions with their reason and permit editing in place. Measure recovery time and successful corrections in the pilot, not only whether people turned the feature on.

#### Read the mechanism

**uncertainty UX**

Uncertainty UX communicates confidence, limits, and alternatives so users calibrate trust. Hiding uncertainty causes overreliance on wrong answers.

Show 'I'm not sure—here are sources' instead of definitive tone on weak retrieval.

**citations**

Citations link UI claims to source passages users can verify. They must be accurate, clickable, and adjacent to the supported statement.

Refund policy answer includes link jumping to handbook section 4.2.

**correction**

Correction flows let users fix wrong AI outputs and feed improvements—labels, prompts, or models. Without correction, errors repeat silently.

Thumbs-down on answer captures expected response for eval set addition.

**undo**

Undo reverses AI-initiated or AI-assisted actions within a safe window. It is essential when actions affect user data or send communications.

Auto-drafted email can be undone for 30 seconds before SMTP send.

**accessibility**

Accessibility ensures AI features work with screen readers, keyboard navigation, and assistive tech—not only visual chat UIs.

Streaming tokens must announce sensibly; citation links need accessible labels.

#### Worked example

**Situation:** New hires trust the assistant's confident tone; a wrong access grant is hard to undo.

**Baseline:** Chat bubble streams answer with no preview or undo.

**Application:** Prototype high-risk flow: show policy evidence preview, require explicit approval, offer undo window, surface uncertainty, log corrections for feedback.

**Test cases:** (1) Normal: low-risk FAQ with citation. (2) Boundary: medium-risk suggestion needing confirm. (3) Adversarial: user rapidly confirms without reading preview.

**Measurement:** Mistake rate, time-on-preview, undo usage, accessibility audit score.

**Design question:** What UX pattern reduces irreversible confirmations without blocking flow entirely?

**Predict before running.** Trust grows from control, evidence, and recoverability—not from confident prose. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 9.5: Human-Centered AI UX"""

CHAPTER = "9.5"
print("chapter hook:", CHAPTER)
risk = "high"
ux = {"preview": True, "approval": risk == "high", "undo_sec": 30 if risk == "high" else 0}
print(ux)
print("inspect step", 1)
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Prototype a high-risk action flow with preview and approval.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why does trust come from control and recoverability?
2. How should citations appear in high-risk flows?
3. What UX baseline maximizes fluent prose only?

<details><summary>Check your reasoning</summary>

Q1: Users trust systems they can verify and reverse. Q2: Inline evidence beside action buttons, not footnotes after confirm. Q3: Streaming confident text with one-click accept.

</details>


### Lecture 06 · Experiments, Adoption, and Value

Measure task success, time, correction effort, retention, adoption, cost, risk, and ROI through staged experiments.

#### Publish the denominator

The team reports 30 assistant-generated changes. Twenty were abandoned or rewritten after review. Nadia wants to know whether total effort changed.

Include failed attempts and rework in the comparison. Use realistic tasks that cover experience levels and task sizes, then measure accepted changes per reviewer hour and defects discovered after merge.

#### Read the mechanism

**A/B testing**

A/B testing compares product variants on live users with ethical guardrails and pre-registered metrics.

Test copilot placement in workflow A versus B measuring task completion time.

**task success**

Task success measures whether users completed their intended job with acceptable quality—not click-through on AI features.

User submitted correct expense report without support contact counts as success.

**adoption**

Adoption tracks who uses the feature, how often, and whether usage persists after novelty fades.

80% weekly active support agents using suggest-reply after 60 days indicates adoption.

**ROI**

ROI compares value gained—time saved, revenue, deflected tickets—to total cost—build, inference, review, incidents.

Saving 500 agent-hours/month at $40/hr must exceed inference plus maintenance cost.

**build versus buy**

Build versus buy weighs custom AI development against vendor APIs and platforms on control, cost, and time-to-value.

Buy GPT-4 API for prototype; build fine-tuned model when volume makes unit economics favorable.

#### Worked example

**Situation:** Leadership asks whether the onboarding assistant improved time-to-productive or merely added AI chrome.

**Baseline:** Launch to 100% users with no measurement plan.

**Application:** Design staged rollout with guardrails, A/B on task success and correction effort, track cost per successful onboarding, define stop thresholds for harm metrics.

**Test cases:** (1) Normal: stable improvement in pilot. (2) Boundary: metric improves for US not EU slice. (3) Adversarial: users skip steps faster by ignoring warnings—false time gain.

**Measurement:** Task success, correction rate, retention, $/success, incident count vs control.

**Design question:** Which metric would stop rollout despite positive average time savings?

**Predict before running.** A technically impressive feature is not successful until it improves a valued workflow. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 9.6: Experiments, Adoption, and Value"""

CHAPTER = "9.6"
print("chapter hook:", CHAPTER)
metrics = {"time_saved": 0.15, "compliance_errors": 0.02, "slice_EU_success": -0.05}
gates = {"compliance_errors_max": 0.01, "slice_min_success": 0.0}
blocked = metrics["compliance_errors"] > gates["compliance_errors_max"]
blocked |= metrics["slice_EU_success"] < gates["slice_min_success"]
print({"blocked": blocked, "reason": "compliance or slice harm"})
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Design a rollout with guardrails and decision thresholds.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why is impressive tech not successful until workflow value improves?
2. How can faster completion indicate harm?
3. What rollout baseline skips guardrails?

<details><summary>Check your reasoning</summary>

Q1: Value is measured in user outcomes and risk, not features. Q2: Users skip safety steps—time down, errors up. Q3: Big-bang release with vanity metrics only.

</details>


**Section checkpoint.** Name a user-facing fallback and a guardrail metric for a pilot.


## Course workshop

Add an out-of-scope case. Decide what an acceptable handoff looks like and check that the demonstration follows it.

Run the reference implementation, then run the checks below it.


In [ ]:
"""Turn acceptance examples into an executable contract.

Spec-driven discipline: write CASES before changing assistant() logic.
Tooling: see docs/getting-started/spec-driven-workflow.md for OpenSpec and Cursor workflows.
Run: python docs/code-samples/09-spec-driven-development.py
"""
SPEC = [
    {"input": "Known answer", "evidence": ["Known answer"], "must_contain": "Known answer"},
    {"input": "Unknown", "evidence": [], "must_contain": "insufficient evidence"},
]


def assistant(case):
    if not case["evidence"]:
        return "insufficient evidence"
    return case["evidence"][0]


failures = []
for case in SPEC:
    actual = assistant(case)
    passed = case["must_contain"].lower() in actual.lower()
    print("PASS" if passed else "FAIL", case["input"], "->", actual)
    if not passed:
        failures.append(case)
assert (1 if failures else 0) == 0


### Verify and extend

Run the assertions. Then change one assumption and add a check for the predicted behavior.


In [ ]:
assert all(case["must_contain"].lower() in assistant(case).lower() for case in SPEC)
assert assistant({"input": "Out of scope", "evidence": []}) == "insufficient evidence"
print('Workshop checks passed')


## Course project

Take one AI feature from problem discovery through a specification-driven implementation and evidence-based release.

Keep your baseline, results, and one failure case so someone else can review the decision you made.
